# Supply Chain Analytics — Python EDA
### End-to-End Distribution & Warehouse Performance Intelligence
**Author:** Sasikumar 2004 | [GitHub](https://github.com/sasikumar2004)

This notebook performs full exploratory data analysis on 8,523 supply chain transactions,
covering data cleaning, KPI extraction, and business-insight visualizations.

**Dataset:** `data/supply_chain_data.csv` — 12 columns, 8,523 rows


---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Clean, light professional theme — renders correctly on GitHub and in Jupyter
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : 'white',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.family'      : 'DejaVu Sans',
    'axes.titlepad'    : 14,
})

# Brand color palette
C_BLUE   = '#2563EB'
C_TEAL   = '#0D9488'
C_AMBER  = '#D97706'
C_RED    = '#DC2626'
C_GREEN  = '#16A34A'
C_PURPLE = '#7C3AED'
PALETTE  = [C_BLUE, C_TEAL, C_AMBER, C_RED, C_GREEN, C_PURPLE,
            '#EA580C', '#0891B2', '#65A30D', '#9333EA']

print('Libraries loaded ✓')


---
## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('data/supply_chain_data.csv')

# Standardise column names
df.columns = df.columns.str.strip().str.lower()

# Rename to business-friendly labels
col_map = {
    'product_category'   : 'Product_Category',
    'product_identifier' : 'Product_ID',
    'product_type'       : 'Product_Type',
    'warehouse_setup_year': 'Warehouse_Year',
    'warehouse_identifier': 'Warehouse_ID',
    'hub_location_tier'  : 'Hub_Tier',
    'warehouse_size'     : 'Warehouse_Size',
    'warehouse_type'     : 'Warehouse_Type',
    'product_visibility' : 'Product_Visibility',
    'product_weight'     : 'Product_Weight',
    'sales'              : 'Revenue',
    'rating'             : 'Rating',
}
df.rename(columns=col_map, inplace=True)

print(f'Shape: {df.shape}')
df.head()


In [ ]:
# Quick dataset summary
print('=' * 55)
print('  SUPPLY CHAIN DATASET — BASIC INFO')
print('=' * 55)
print(f'  Total Records     : {len(df):,}')
print(f'  Total Columns     : {df.shape[1]}')
print(f'  Total Revenue (₹) : {df["Revenue"].sum():,.2f}')
print(f'  Avg Revenue       : {df["Revenue"].mean():.2f}')
print(f'  Unique Products   : {df["Product_ID"].nunique()}')
print(f'  Unique Warehouses : {df["Warehouse_ID"].nunique()}')
print(f'  Unique Prod Types : {df["Product_Type"].nunique()}')
print(f'  Year Range        : {df["Warehouse_Year"].min()} – {df["Warehouse_Year"].max()}')
print('=' * 55)


In [ ]:
df.info()


In [ ]:
df.describe().round(2)


---
## 3. Data Cleaning

We identify and fix three issues: (1) inconsistent `Product_Category` casing,
(2) missing `Product_Weight` values (imputed by group median), and
(3) missing `Warehouse_Size` values (filled with mode).


In [ ]:
# Missing value summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])


In [ ]:
# Fix Product_Category inconsistencies
print('Before:', df['Product_Category'].unique())
df['Product_Category'] = df['Product_Category'].replace({
    'low fat': 'Low Fat', 'LF': 'Low Fat', 'Low fat': 'Low Fat',
    'reg'    : 'Regular',  'REGULAR': 'Regular'
})
print('After :', df['Product_Category'].unique())

# Assertion: exactly 2 categories
assert df['Product_Category'].nunique() == 2, 'ERROR: Expected exactly 2 product categories!'
print('Assertion passed: 2 product categories ✓')


In [ ]:
# Impute missing Product_Weight — group median by Product_Type
df['Product_Weight'] = df.groupby('Product_Type')['Product_Weight'].transform(
    lambda x: x.fillna(x.median())
)
# Fill any remaining (products with all-null group) with overall median
df['Product_Weight'].fillna(df['Product_Weight'].median(), inplace=True)

# Fill missing Warehouse_Size with mode
df['Warehouse_Size'].fillna(df['Warehouse_Size'].mode()[0], inplace=True)

# Assertions: no nulls remain in critical columns
assert df['Product_Weight'].isnull().sum() == 0, 'ERROR: Nulls remain in Product_Weight!'
assert df['Warehouse_Size'].isnull().sum() == 0, 'ERROR: Nulls remain in Warehouse_Size!'

print(f'Total nulls after cleaning: {df.isnull().sum().sum()}')
print('All cleaning assertions passed ✓')


---
## 4. KPI Summary — Key Metrics

Four headline KPIs give an instant read on overall supply chain health.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Supply Chain Analytics — KPI Dashboard',
             fontsize=15, fontweight='bold', y=1.03)

kpis = [
    ('Total Revenue',    f"₹ {df['Revenue'].sum()/1_000_000:.2f} M", C_BLUE),
    ('Avg Transaction',  f"₹ {df['Revenue'].mean():.0f}",             C_TEAL),
    ('Total Shipments',  f"{len(df):,}",                               C_AMBER),
    ('Avg Satisfaction', f"{df['Rating'].mean():.2f} / 5",            C_GREEN),
]
for ax, (label, value, color) in zip(axes, kpis):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    ax.add_patch(plt.Rectangle((0,0), 1, 1, facecolor='#F8FAFC',
                 edgecolor=color, linewidth=2.5, transform=ax.transAxes))
    ax.add_patch(plt.Rectangle((0, 0.88), 1, 0.12, facecolor=color,
                 transform=ax.transAxes))
    ax.text(0.5, 0.6, value, ha='center', va='center', fontsize=21,
            fontweight='bold', color='#1E293B', transform=ax.transAxes)
    ax.text(0.5, 0.32, label, ha='center', va='center', fontsize=11,
            color='#475569', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('docs/kpi_cards.png', dpi=150, bbox_inches='tight')
plt.show()
print('KPI dashboard saved to docs/kpi_cards.png ✓')


---
## 5. Revenue by Product Type

Fruits & Vegetables and Snack Foods lead all 16 product types, together accounting
for roughly 35% of total revenue. This signals strong demand for fresh produce and
impulse-buy categories — both require different supply chain strategies (cold-chain
vs ambient storage).


In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

pt_rev = df.groupby('Product_Type')['Revenue'].sum().sort_values() / 1000
colors = [C_BLUE if v == pt_rev.max() else '#93C5FD' for v in pt_rev.values]

bars = ax.barh(pt_rev.index, pt_rev.values, color=colors, edgecolor='none', height=0.7)

for bar, val in zip(bars, pt_rev.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'₹{val:.1f}K', va='center', fontsize=9, color='#334155')

ax.set_xlabel('Total Revenue (₹ Thousands)', fontsize=11)
ax.set_title('Revenue by Product Type', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.savefig('docs/revenue_by_product.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 6. Revenue by Warehouse Type & Size

**Supermarket Type 1** dominates with ~65% of revenue — its combination of high
foot traffic and broad product range makes it the backbone of the network.
**Medium-sized** warehouses generate the most total revenue, suggesting they hit
the optimal balance between capacity and operational cost.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Warehouse Performance Analysis', fontsize=14, fontweight='bold')

# Revenue by Warehouse Type (bar)
wt_rev = df.groupby('Warehouse_Type')['Revenue'].sum().sort_values(ascending=False) / 1000
bar_colors = [C_BLUE, C_TEAL, C_AMBER, C_RED]
bars = axes[0].bar(range(len(wt_rev)), wt_rev.values, color=bar_colors[:len(wt_rev)],
                   edgecolor='none', width=0.6)
for bar, val in zip(bars, wt_rev.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'₹{val:.0f}K', ha='center', fontsize=9, color='#334155')
axes[0].set_xticks(range(len(wt_rev)))
axes[0].set_xticklabels(wt_rev.index, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('Revenue (₹ Thousands)', fontsize=11)
axes[0].set_title('Revenue by Warehouse Type')

# Revenue by Warehouse Size (pie)
ws_rev = df.groupby('Warehouse_Size')['Revenue'].sum()
wedges, texts, autotexts = axes[1].pie(
    ws_rev.values, labels=ws_rev.index,
    autopct='%1.1f%%', colors=[C_BLUE, C_TEAL, C_AMBER][:len(ws_rev)],
    startangle=90, pctdistance=0.78,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts: at.set_fontsize(10); at.set_fontweight('bold')
axes[1].set_title('Revenue Share by Warehouse Size')

plt.tight_layout()
plt.savefig('docs/warehouse_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 7. Revenue by Distribution Zone (Hub Tier)

**Tier 3** locations lead in total revenue (~39%), reflecting the high transaction
volume in these mass-market zones. However, **Tier 1** shows the highest *average*
revenue per shipment — premium locations serve higher-value orders.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

tier_rev = df.groupby('Hub_Tier')['Revenue'].sum() / 1000
tier_avg = df.groupby('Hub_Tier')['Revenue'].mean()

x = np.arange(len(tier_rev))
w = 0.38

b1 = ax.bar(x - w/2, tier_rev.values, width=w, label='Total Revenue (K)', color=C_BLUE, edgecolor='none')
ax2 = ax.twinx()
b2 = ax2.bar(x + w/2, tier_avg.values, width=w, label='Avg Revenue / Shipment', color=C_AMBER, alpha=0.85, edgecolor='none')

ax.set_xticks(x)
ax.set_xticklabels(tier_rev.index, fontsize=12)
ax.set_ylabel('Total Revenue (₹ Thousands)', color=C_BLUE, fontsize=11)
ax2.set_ylabel('Avg Revenue per Shipment (₹)', color=C_AMBER, fontsize=11)
ax.set_title('Revenue by Distribution Zone — Hub Tier Analysis', fontsize=14, fontweight='bold')

lines = [b1, b2]
labels = ['Total Revenue (K)', 'Avg Revenue / Shipment']
ax.legend(lines, labels, loc='upper right')

plt.tight_layout()
plt.savefig('docs/hub_tier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 8. Year-wise Warehouse Revenue Trend

Revenue dips around 2014–2015, then recovers steadily. The notable **2018 spike**
(warehouses established in 2018 outperform the average by ~57%) warrants deeper
investigation — a likely operational change or network expansion drove this.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

yr_rev = df.groupby('Warehouse_Year')['Revenue'].agg(['sum', 'count']).reset_index()
yr_rev['sum'] /= 1000

ax.fill_between(yr_rev['Warehouse_Year'], yr_rev['sum'], alpha=0.15, color=C_BLUE)
ax.plot(yr_rev['Warehouse_Year'], yr_rev['sum'],
        color=C_BLUE, linewidth=2.5, marker='o', markersize=7,
        markerfacecolor=C_AMBER, markeredgecolor=C_BLUE, markeredgewidth=1.5)

for _, row in yr_rev.iterrows():
    ax.annotate(f"₹{row['sum']:.0f}K",
                (row['Warehouse_Year'], row['sum']),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=8, color='#475569')

ax.set_xlabel('Warehouse Establishment Year', fontsize=11)
ax.set_ylabel('Total Revenue (₹ Thousands)', fontsize=11)
ax.set_title('Year-wise Warehouse Revenue Trend', fontsize=14, fontweight='bold')
ax.set_xticks(yr_rev['Warehouse_Year'])
ax.grid(alpha=0.4)

plt.tight_layout()
plt.savefig('docs/year_trend.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 9. Correlation Matrix

The low correlations across all numeric features confirm that **no single variable
linearly drives revenue**. Product visibility, weight, and rating are largely
independent predictors — suggesting warehouse-level factors (type, tier, size)
are stronger revenue levers than individual product attributes.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

numeric = df[['Product_Weight', 'Product_Visibility', 'Revenue', 'Rating']].copy()
corr = numeric.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlBu_r', center=0, ax=ax,
    annot_kws={'size': 12},
    linecolor='white', linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('docs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 10. Revenue Distribution Analysis

Revenue follows a roughly **normal distribution** with a slight right skew — most
transactions fall in the ₹80–₹200 range. The scatter plot confirms the **visibility
paradox**: items at extreme visibility values (very low or very high) don't
outperform average-visibility products.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Revenue Distribution Analysis', fontsize=14, fontweight='bold')

# Histogram
axes[0].hist(df['Revenue'], bins=40, color=C_BLUE, edgecolor='white', alpha=0.85)
axes[0].axvline(df['Revenue'].mean(),   color=C_AMBER, linestyle='--', lw=2,
                label=f"Mean: ₹{df['Revenue'].mean():.0f}")
axes[0].axvline(df['Revenue'].median(), color=C_GREEN, linestyle='--', lw=2,
                label=f"Median: ₹{df['Revenue'].median():.0f}")
axes[0].set_title('Revenue Distribution')
axes[0].set_xlabel('Revenue (₹)')
axes[0].legend(fontsize=9)

# Box plot by Product Category
cat_groups = [df[df['Product_Category'] == c]['Revenue'].values
              for c in df['Product_Category'].unique()]
bp = axes[1].boxplot(cat_groups, patch_artist=True,
                     labels=df['Product_Category'].unique(),
                     medianprops={'color': C_AMBER, 'linewidth': 2.5})
for patch, color in zip(bp['boxes'], [C_BLUE, C_TEAL]):
    patch.set_facecolor(color); patch.set_alpha(0.6)
axes[1].set_title('Revenue by Product Category')
axes[1].set_ylabel('Revenue (₹)')

# Visibility vs Revenue scatter
axes[2].scatter(df['Product_Visibility'], df['Revenue'],
                alpha=0.25, color=C_TEAL, s=12, edgecolors='none')
axes[2].set_xlabel('Product Visibility (0–1)')
axes[2].set_ylabel('Revenue (₹)')
axes[2].set_title('Visibility vs Revenue')
corr_val = df['Product_Visibility'].corr(df['Revenue'])
axes[2].text(0.05, 0.92, f'r = {corr_val:.3f}', transform=axes[2].transAxes,
             fontsize=10, color='#475569')

plt.tight_layout()
plt.savefig('docs/distribution_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 11. Top Product Analysis & Satisfaction Scores

The top 10 product types by revenue are largely staples and household essentials —
confirming this is a **daily-needs** supply chain. Satisfaction ratings show low
variance (~3.9–4.1 across types), suggesting systemic rather than product-specific
satisfaction drivers (delivery speed, packaging, availability).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Top Product Analysis', fontsize=14, fontweight='bold')

# Top 10 by revenue
top10 = df.groupby('Product_Type')['Revenue'].sum().sort_values(ascending=False).head(10)
colors_t = [C_BLUE if i == 0 else '#93C5FD' for i in range(10)]
axes[0].barh(top10.index[::-1], top10.values[::-1] / 1000,
             color=colors_t[::-1], edgecolor='none', height=0.65)
axes[0].set_title('Top 10 Product Types by Revenue')
axes[0].set_xlabel('Revenue (₹ Thousands)')

# Average satisfaction by Product Type (top 10)
avg_rating = df.groupby('Product_Type')['Rating'].mean().sort_values(ascending=False).head(10)
global_avg = df['Rating'].mean()
bar_colors_r = [C_GREEN if r >= global_avg else C_AMBER for r in avg_rating.values]
axes[1].bar(range(len(avg_rating)), avg_rating.values, color=bar_colors_r, edgecolor='none', width=0.6)
axes[1].axhline(global_avg, color=C_RED, linestyle='--', lw=1.5,
                label=f'Global Avg: {global_avg:.2f}')
axes[1].set_xticks(range(len(avg_rating)))
axes[1].set_xticklabels(avg_rating.index, rotation=35, ha='right', fontsize=8)
axes[1].set_ylabel('Average Rating')
axes[1].set_title('Avg Satisfaction Rating by Product Type')
axes[1].set_ylim(3.5, 4.5)
axes[1].legend()

plt.tight_layout()
plt.savefig('docs/top_products.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 12. Final Summary Report

In [ ]:
print('=' * 60)
print('   SUPPLY CHAIN ANALYTICS — FINAL SUMMARY REPORT')
print('=' * 60)

print(f'\n  DATASET')
print(f'  Total Records      : {len(df):,}')
print(f'  Features           : {df.shape[1]}')
print(f'  Year Range         : {df["Warehouse_Year"].min()} – {df["Warehouse_Year"].max()}')

print(f'\n  REVENUE KPIs')
print(f'  Total Revenue      : ₹ {df["Revenue"].sum():,.2f}')
print(f'  Avg per Shipment   : ₹ {df["Revenue"].mean():.2f}')
print(f'  Max Transaction    : ₹ {df["Revenue"].max():.2f}')
print(f'  Std Deviation      : ₹ {df["Revenue"].std():.2f}')

print(f'\n  WAREHOUSE INSIGHTS')
print(f'  Best Warehouse Type: {df.groupby("Warehouse_Type")["Revenue"].sum().idxmax()}')
print(f'  Top Revenue Zone   : {df.groupby("Hub_Tier")["Revenue"].sum().idxmax()}')
print(f'  Top Warehouse Size : {df.groupby("Warehouse_Size")["Revenue"].sum().idxmax()}')

print(f'\n  PRODUCT INSIGHTS')
print(f'  Top Product Type   : {df.groupby("Product_Type")["Revenue"].sum().idxmax()}')
print(f'  Top Category       : {df.groupby("Product_Category")["Revenue"].sum().idxmax()}')

print(f'\n  SATISFACTION')
print(f'  Avg Rating         : {df["Rating"].mean():.3f} / 5.0')
print(f'  Best Rated Type    : {df.groupby("Product_Type")["Rating"].mean().idxmax()}')

print('\n' + '=' * 60)
print('  All charts saved to docs/ folder')
print('  Analysis complete ✓')
print('=' * 60)
